# Testing Agents and RAG

This notebook tests three components:
1. Simple Agent - basic tool-using agent
2. Agent with Helpfulness - agent with helpfulness evaluation loop
3. RAG Tool - Retrieval Augmented Generation tool


In [6]:
# Setup and Imports
import os
import getpass
from dotenv import load_dotenv

# Load environment variables from .env file if it exists
load_dotenv()

# Set up Together API key if not already set
if not os.environ.get("TOGETHER_API_KEY"):
    os.environ["TOGETHER_API_KEY"] = getpass.getpass("Enter your Together API key: ")

# Optional: Set model endpoint (defaults to openai/gpt-oss-20b)
if not os.environ.get("TOGETHER_MODEL"):
    os.environ["TOGETHER_MODEL"] = "openai/gpt-oss-20b"

# Import agent graphs and utilities
from app.graphs.simple_agent import graph as simple_agent_graph
from app.graphs.agent_with_helpfulness import graph as helpfulness_agent_graph
from app.rag import retrieve_information
from app.state import AgentState
from langchain_core.messages import HumanMessage

print("✓ Setup complete!")
print(f"✓ Using model: {os.environ.get('TOGETHER_MODEL', 'openai/gpt-oss-20b')}")
print(f"✓ API key set: {'Yes' if os.environ.get('TOGETHER_API_KEY') else 'No'}")


✓ Setup complete!
✓ Using model: openai/gpt-oss-20b
✓ API key set: Yes


## Test 1: Simple Agent

Testing the basic tool-using agent with different types of queries.


In [7]:
# Test 1.1: Simple query (no tools needed)
print("=" * 60)
print("Test 1.1: Simple Query - 'What is LangGraph?'")
print("=" * 60)

state: AgentState = {
    "messages": [HumanMessage(content="What is LangGraph?")]
}

result = simple_agent_graph.invoke(state)

print("\nAgent Response:")
print(result["messages"][-1].content)
print(f"\nTotal messages in conversation: {len(result['messages'])}")


Test 1.1: Simple Query - 'What is LangGraph?'

Agent Response:
LangGraph is a library within the LangChain ecosystem designed for creating, managing, and executing complex AI workflows using graph-based structures. It allows developers to define nodes, which represent units of work such as interacting with language models, calling APIs, or performing data manipulation, and connect them with edges that define the flow of execution and data. LangGraph supports cyclic graphs, enabling iterative and stateful processes, making it suitable for building sophisticated multi-agent systems and AI applications. It is inspired by graph processing frameworks like Pregel, Apache Beam, and NetworkX, and can be used independently of LangChain.

Total messages in conversation: 4


In [8]:
# Test 1.2: Query that might trigger tool usage
print("=" * 60)
print("Test 1.2: Tool-Using Query - 'Search for recent papers about transformers'")
print("=" * 60)

state: AgentState = {
    "messages": [HumanMessage(content="Search for recent papers about transformer architectures")]
}

result = simple_agent_graph.invoke(state)

print("\nFull Conversation Flow:")
print("-" * 60)
for i, msg in enumerate(result["messages"]):
    print(f"\nMessage {i+1}: {type(msg).__name__}")
    if hasattr(msg, 'content') and msg.content:
        print(f"Content: {msg.content[:200]}...")
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"Tool calls: {[tc.get('name', 'unknown') for tc in msg.tool_calls]}")
print("\n" + "=" * 60)
print("Final Response:")
print(result["messages"][-1].content)


Test 1.2: Tool-Using Query - 'Search for recent papers about transformers'

Full Conversation Flow:
------------------------------------------------------------

Message 1: HumanMessage
Content: Search for recent papers about transformer architectures...

Message 2: AIMessage
Tool calls: ['arxiv', 'tavily_search_results_json']

Message 3: ToolMessage
Content: Error: arxiv.HTTPError('https://export.arxiv.org/api/query?search_query=recent+papers+about+transformer+architectures&id_list=&sortBy=relevance&sortOrder=descending&start=0&max_results=100', 3, 429)
 ...

Message 4: ToolMessage
Content: [{"title": "Advances in Transformer Architectures: Integrating BERT, GPT, and ...", "url": "https://papers.ssrn.com/sol3/papers.cfm?abstract_id=5368602", "content": "Advances in Transformer Architectu...

Message 5: AIMessage
Content: I found several recent papers related to transformer architectures. Here are some of the notable ones:

1. [Advances in Transformer Architectures: Integrating BERT, G

## Test 2: Agent with Helpfulness

Testing the agent that includes a helpfulness evaluation loop.


In [9]:
# Test 2.1: Query that should be helpful
print("=" * 60)
print("Test 2.1: Helpful Query - 'Explain what RAG is'")
print("=" * 60)

state: AgentState = {
    "messages": [HumanMessage(content="Explain what RAG is")]
}

result = helpfulness_agent_graph.invoke(state)

print("\nConversation Flow:")
print("-" * 60)
for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__
    content = getattr(msg, 'content', '')[:150] if hasattr(msg, 'content') else ''
    print(f"{i+1}. {msg_type}: {content}...")
    
    # Highlight helpfulness evaluations
    if hasattr(msg, 'content') and 'HELPFULNESS' in str(msg.content):
        print(f"   ⚠️  Helpfulness check: {msg.content}")

print("\n" + "=" * 60)
print("Final Response:")
print(result["messages"][-1].content)


Test 2.1: Helpful Query - 'Explain what RAG is'

Conversation Flow:
------------------------------------------------------------
1. HumanMessage: Explain what RAG is...
2. AIMessage: ...
3. ToolMessage: I don't know....
4. AIMessage: RAG typically stands for Retrieval-Augmented Generation, which is a technique used in natural language processing. It combines traditional language mo...
5. AIMessage: HELPFULNESS:Y...
   ⚠️  Helpfulness check: HELPFULNESS:Y

Final Response:
HELPFULNESS:Y


In [10]:
# Test 2.2: Query that might need iteration
print("=" * 60)
print("Test 2.2: Complex Query - 'What are the latest developments in AI?'")
print("=" * 60)

state: AgentState = {
    "messages": [HumanMessage(content="What are the latest developments in AI?")]
}

result = helpfulness_agent_graph.invoke(state)

print("\nConversation Flow (showing helpfulness checks):")
print("-" * 60)
helpfulness_checks = []
for i, msg in enumerate(result["messages"]):
    if hasattr(msg, 'content') and 'HELPFULNESS' in str(msg.content):
        helpfulness_checks.append((i, msg.content))
        print(f"Message {i+1}: {msg.content}")

print(f"\nTotal helpfulness evaluations: {len(helpfulness_checks)}")
print(f"Total messages: {len(result['messages'])}")
print("\n" + "=" * 60)
print("Final Response:")
print(result["messages"][-1].content)


Test 2.2: Complex Query - 'What are the latest developments in AI?'

Conversation Flow (showing helpfulness checks):
------------------------------------------------------------
Message 6: HELPFULNESS:Y

Total helpfulness evaluations: 1
Total messages: 6

Final Response:
HELPFULNESS:Y


## Test 3: RAG Tool

Testing the Retrieval Augmented Generation tool directly and through the agent.


In [13]:
# Test 3.1: Test RAG tool directly
print("=" * 60)
print("Test 3.1: RAG Tool Direct - Query about AI usage")
print("=" * 60)

query = "How are people using AI in their daily work?"
print(f"Query: {query}\n")

try:
    result = retrieve_information(query)
    print("RAG Response:")
    print(result)
except Exception as e:
    print(f"Error: {e}")
    print("\nNote: Make sure you have PDF files in the 'data' directory")
    print("and that OpenAI API key is set for embeddings.")


Test 3.1: RAG Tool Direct - Query about AI usage
Query: How are people using AI in their daily work?

RAG Response:
People use generative‑AI tools like ChatGPT in a wide range of day‑to‑day work activities, and the most common ways are summarized in the study:

| What users do with AI | Typical work‑related activity |
|-----------------------|--------------------------------|
| **Ask for information or clarification** (≈ 49 % of all messages) | Search, fact‑checking, policy or rule comprehension, quick fact lookup – essentially “obtaining, documenting, and interpreting information.” |
| **Perform a task (Doing)** (≈ 40 % of all messages, but 56 % of work‑related ones) | <ul><li>**Writing** – drafting emails, reports, proposals, meeting notes, marketing copy, code comments, or even entire documents (≈ ¾ of the “Doing” share).  <li>**Programming** – writing, debugging, or explaining short code snippets (≈ 4 % of all messages).  <li>**Data‑analysis tasks** – generating formulas, explainin

In [12]:
# Test 3.2: Test RAG through simple agent (RAG is in the tool belt)
print("=" * 60)
print("Test 3.2: RAG via Simple Agent")
print("=" * 60)

state: AgentState = {
    "messages": [HumanMessage(content="Use the retrieve_information tool to find out how people are using AI in their work")]
}

result = simple_agent_graph.invoke(state)

print("\nAgent Conversation:")
print("-" * 60)
for i, msg in enumerate(result["messages"]):
    msg_type = type(msg).__name__
    if hasattr(msg, 'tool_calls') and msg.tool_calls:
        print(f"{i+1}. {msg_type}: Called tools: {[tc.get('name', 'unknown') for tc in msg.tool_calls]}")
    elif hasattr(msg, 'content'):
        content = msg.content[:200] if len(msg.content) > 200 else msg.content
        print(f"{i+1}. {msg_type}: {content}")

print("\n" + "=" * 60)
print("Final Response:")
print(result["messages"][-1].content)


Test 3.2: RAG via Simple Agent

Agent Conversation:
------------------------------------------------------------
1. HumanMessage: Use the retrieve_information tool to find out how people are using AI in their work
2. AIMessage: Called tools: ['retrieve_information']
3. ToolMessage: **How people are using AI (particularly ChatGPT) in their work**

| Intent | How it shows up in work conversations | Typical content / tasks | Share of work‑related messages |
|--------|--------------
4. AIMessage: People are using AI in their work primarily to generate written content such as emails, reports, and proposals, which accounts for the majority of "Doing" tasks. They also use AI to ask questions, see

Final Response:
People are using AI in their work primarily to generate written content such as emails, reports, and proposals, which accounts for the majority of "Doing" tasks. They also use AI to ask questions, seek clarifications, and gather information to inform their decisions. Additionally, AI

## Test 4: Comparison Test

Running the same query through all three approaches to compare behavior.


In [11]:
# Comparison: Same query through all three approaches
test_query = "What is RAG and how does it work?"

print("=" * 60)
print("COMPARISON TEST")
print("=" * 60)
print(f"Query: {test_query}\n")

# Test with Simple Agent
print("\n" + "-" * 60)
print("1. SIMPLE AGENT")
print("-" * 60)
state1: AgentState = {"messages": [HumanMessage(content=test_query)]}
result1 = simple_agent_graph.invoke(state1)
print(f"Response length: {len(result1['messages'][-1].content)} chars")
print(f"Total messages: {len(result1['messages'])}")
print(f"Response preview: {result1['messages'][-1].content[:200]}...")

# Test with Helpfulness Agent
print("\n" + "-" * 60)
print("2. AGENT WITH HELPFULNESS")
print("-" * 60)
state2: AgentState = {"messages": [HumanMessage(content=test_query)]}
result2 = helpfulness_agent_graph.invoke(state2)
print(f"Response length: {len(result2['messages'][-1].content)} chars")
print(f"Total messages: {len(result2['messages'])}")
helpfulness_count = sum(1 for m in result2['messages'] if 'HELPFULNESS' in str(getattr(m, 'content', '')))
print(f"Helpfulness checks: {helpfulness_count}")
print(f"Response preview: {result2['messages'][-1].content[:200]}...")

# Test with RAG Tool directly
print("\n" + "-" * 60)
print("3. RAG TOOL DIRECT")
print("-" * 60)
try:
    result3 = retrieve_information(test_query)
    print(f"Response length: {len(str(result3))} chars")
    print(f"Response preview: {str(result3)[:200]}...")
except Exception as e:
    print(f"Error: {e}")

print("\n" + "=" * 60)
print("Comparison complete!")
print("=" * 60)


COMPARISON TEST
Query: What is RAG and how does it work?


------------------------------------------------------------
1. SIMPLE AGENT
------------------------------------------------------------
Response length: 654 chars
Total messages: 4
Response preview: RAG, or Retrieval-Augmented Generation, is an advanced technique used in natural language processing that combines the strengths of retrieval-based methods and generative models. It works by first ret...

------------------------------------------------------------
2. AGENT WITH HELPFULNESS
------------------------------------------------------------
Response length: 13 chars
Total messages: 5
Helpfulness checks: 1
Response preview: HELPFULNESS:Y...

------------------------------------------------------------
3. RAG TOOL DIRECT
------------------------------------------------------------


/var/folders/2k/24tggmp93z152ndxyqxtq4q80000gn/T/ipykernel_65245/1759346965.py:36: LangChainDeprecationWarning: The method `BaseTool.__call__` was deprecated in langchain-core 0.1.47 and will be removed in 1.0. Use :meth:`~invoke` instead.
  result3 = retrieve_information(test_query)


Response length: 12 chars
Response preview: I don't know...

Comparison complete!
